# Real Estate Appreciation Prediction Pipeline — v2
**Sponsor: First American Title™ — Best Use of AI in Real Estate**

Pipeline: **Zillow ZHVI** (ZIP-level monthly index) + **FRED API** (macro) + **Census ACS** (ZIP demographics) → Repeat-Sale Feature Engineering → XGBoost (A100 GPU)

### Why v2?
The original pipeline relied on ATTOM's `/saleshistory/detail` endpoint, which returned **401 Unauthorized** on the free trial tier. As a result:
- `baseline_sale_price` and `assessed_value` were **100% null**
- The model fell back to using `living_area_sqft` as a price proxy
- Sale dates were generated from MD5 hashes of `attomid`, producing fake temporal spread
- Training targets were appreciation of *square footage* over *synthetic time* — meaningless
- MAPE reached **8–11 million %** confirming the predictions were pure noise

**Fix:** Replace ATTOM entirely with the **Zillow Research ZHVI** dataset (free, no key, 27k ZIPs, monthly back to 2000). Targets are now real home-value appreciation percentages.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q pandas numpy httpx xgboost scikit-learn joblib nest_asyncio requests

In [ ]:
# Cell 2 — Imports + API keys from Colab Secrets
import os
import json
import asyncio
import warnings
import time

import numpy as np
import pandas as pd
import httpx
import requests
import xgboost
import joblib
import nest_asyncio
nest_asyncio.apply()

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')

# ── API keys — ATTOM is no longer used ───────────────────────────────────────
try:
    from google.colab import userdata
    FRED_API_KEY   = userdata.get('FRED_API_KEY')
    CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')   # optional — removes Census rate limits
except Exception:
    FRED_API_KEY   = os.environ.get('FRED_API_KEY', '')
    CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY', '')

assert FRED_API_KEY, (
    'FRED_API_KEY not found in Colab Secrets. '
    'Get a free key at https://fred.stlouisfed.org/docs/api/api_key.html'
)
census_status = 'loaded' if CENSUS_API_KEY else 'not set (optional — will use anonymous Census access)'
print(f'FRED_API_KEY   : loaded')
print(f'CENSUS_API_KEY : {census_status}')

In [ ]:
# Cell 3 — Download Zillow ZHVI and compute appreciation targets
#
# Source : Zillow Research Data — https://www.zillow.com/research/data/
# File   : ZHVI All Homes (SFR, Condo/Co-op), Smoothed, Seasonally Adjusted
# Level  : ZIP code (ZCTA)
# Format : Wide CSV — rows = ZIPs, columns = monthly dates (YYYY-MM-DD)
# Free   : No API key, no account required
#
# If the direct URL below fails (Zillow occasionally rotates CDN paths), download
# the file manually from the link above and upload it to Colab as zhvi_zip.csv.

ZHVI_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

ZHVI_LOCAL = '/content/zhvi_zip.csv'

if not os.path.exists(ZHVI_LOCAL):
    print('Downloading ZHVI CSV (~25 MB)...')
    resp = requests.get(ZHVI_URL, timeout=120)
    if resp.status_code == 200:
        with open(ZHVI_LOCAL, 'wb') as f:
            f.write(resp.content)
        print(f'Saved {ZHVI_LOCAL}')
    else:
        raise RuntimeError(
            f'Direct download failed (HTTP {resp.status_code}).\n'
            f'Download manually from https://www.zillow.com/research/data/ '
            f'and upload to Colab as /content/zhvi_zip.csv'
        )
else:
    print(f'Found cached {ZHVI_LOCAL}')

# ── Load and inspect ──────────────────────────────────────────────────────────
df_zhvi_wide = pd.read_csv(ZHVI_LOCAL)
print(f'\nShape          : {df_zhvi_wide.shape}')
print(f'First 10 cols  : {list(df_zhvi_wide.columns[:10])}')
print(f'Last 5 cols    : {list(df_zhvi_wide.columns[-5:])}')

META_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
             'StateName', 'State', 'City', 'Metro', 'CountyName']
META_COLS = [c for c in META_COLS if c in df_zhvi_wide.columns]
date_cols  = [c for c in df_zhvi_wide.columns if c not in META_COLS]

print(f'Metadata cols  : {META_COLS}')
print(f'Date columns   : {len(date_cols)}  ({date_cols[0]} → {date_cols[-1]})')
print(f'ZIP codes      : {df_zhvi_wide["RegionName"].nunique():,}')

In [ ]:
# Cell 4 — Reshape ZHVI to long format + compute appreciation targets
#
# Target variable: percentage appreciation over N months
#   target = (zhvi_future - zhvi_base) / zhvi_base * 100.0
# Horizons: 6, 12, and 36 months
# Sanity filter: drop rows where appreciation is outside [-50%, +100%]
# Stride: use every 3rd month as a base date to keep dataset size manageable

# ── Melt wide → long ──────────────────────────────────────────────────────────
df_long = df_zhvi_wide.melt(
    id_vars=META_COLS,
    value_vars=date_cols,
    var_name='date',
    value_name='zhvi'
)
df_long['date'] = pd.to_datetime(df_long['date'])
df_long = df_long.dropna(subset=['zhvi'])
df_long = df_long.rename(columns={
    'RegionName': 'zip',
    'State':      'state',
    'CountyName': 'county',
    'City':       'city',
})
df_long['zip'] = df_long['zip'].astype(str).str.zfill(5)
df_long = df_long.sort_values(['zip', 'date']).reset_index(drop=True)

print(f'Long format shape : {df_long.shape}')
print(f'Date range        : {df_long["date"].min()} → {df_long["date"].max()}')
print(f'States covered    : {df_long["state"].nunique()}')
print(f'ZIPs covered      : {df_long["zip"].nunique():,}')

# ── Compute appreciation targets ─────────────────────────────────────────────
HORIZONS      = [6, 12, 36]
STRIDE_MONTHS = 3   # use every 3rd month as base to keep row count ~1–2 M

print('\nComputing appreciation targets (this may take 2–5 minutes)...')
rows = []
groups = df_long.groupby(['zip', 'state', 'county'])
n_groups = len(groups)

for idx, ((zip_code, state, county), grp) in enumerate(groups):
    grp = grp.sort_values('date').reset_index(drop=True)
    for i in range(0, len(grp), STRIDE_MONTHS):
        base_row  = grp.iloc[i]
        base_val  = base_row['zhvi']
        base_date = base_row['date']
        if base_val <= 0:
            continue
        for h in HORIZONS:
            future = grp[grp['date'] >= base_date + pd.DateOffset(months=h)]
            if future.empty:
                continue
            fut_val = future.iloc[0]['zhvi']
            if fut_val <= 0:
                continue
            appreciation_pct = (fut_val - base_val) / base_val * 100.0
            if not (-50.0 <= appreciation_pct <= 100.0):   # sanity bounds
                continue
            rows.append({
                'zip':            zip_code,
                'state':          state,
                'county':         county,
                'date':           base_date,
                'horizon_months': h,
                'baseline_zhvi':  base_val,
                'target':         appreciation_pct,   # percentage points, e.g. 4.2 = 4.2%
            })
    if (idx + 1) % 5000 == 0:
        print(f'  {idx+1:,}/{n_groups:,} ZIPs processed — {len(rows):,} rows so far')

df_targets = pd.DataFrame(rows)
df_targets.to_csv('/content/zhvi_targets.csv', index=False)

print(f'\n✓ Saved /content/zhvi_targets.csv')
print(f'  Shape  : {df_targets.shape}')
print(f'\nTarget distribution (appreciation %):')
print(df_targets['target'].describe().round(2))
print(f'\nHorizon breakdown:')
print(df_targets['horizon_months'].value_counts().sort_index())

In [ ]:
# Cell 5 — Fetch FRED macro indicators
#
# Series fetched (monthly, back to 2000):
#   MORTGAGE30US — 30-year fixed mortgage rate
#   FEDFUNDS     — Federal Funds effective rate
#   UNRATE       — Civilian unemployment rate
#   CPIAUCSL     — CPI all urban consumers (inflation proxy)

FRED_BASE   = 'https://api.stlouisfed.org/fred/series/observations'
FRED_SERIES = {
    'mortgage_rate': 'MORTGAGE30US',
    'fed_funds':     'FEDFUNDS',
    'unemployment':  'UNRATE',
    'CPI':           'CPIAUCSL',
}

async def fetch_fred_series(series_id, observation_start='2000-01-01'):
    params = {
        'series_id':         series_id,
        'api_key':           FRED_API_KEY,
        'file_type':         'json',
        'observation_start': observation_start,
        'frequency':         'm',
    }
    for attempt in range(5):
        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(FRED_BASE, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            df = pd.DataFrame(data.get('observations', []))[['date', 'value']]
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            df['date']  = pd.to_datetime(df['date'])
            return df.rename(columns={'value': series_id})
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [{series_id}] attempt {attempt+1} failed: {e} — retrying in {wait}s')
            await asyncio.sleep(wait)
    return pd.DataFrame(columns=['date', series_id])

fred_dfs = []
for name, sid in FRED_SERIES.items():
    print(f'Fetching {sid} ({name})...')
    df_s = await fetch_fred_series(sid)
    if not df_s.empty:
        fred_dfs.append(df_s.rename(columns={sid: name}).set_index('date'))
        print(f'  → {len(df_s)} observations')
    else:
        print(f'  WARNING: No data returned for {sid}')

df_fred = pd.concat(fred_dfs, axis=1).reset_index().sort_values('date')
df_fred.to_csv('/content/fred_macros.csv', index=False)
print(f'\n✓ Saved /content/fred_macros.csv — shape: {df_fred.shape}')
df_fred.tail()

In [ ]:
# Cell 6 — Fetch Census ACS ZIP-level features
#
# Source : Census Bureau ACS 5-Year Estimates (2022)
# Level  : All ZCTAs (ZIP Code Tabulation Areas) nationally
# Free   : Yes — CENSUS_API_KEY is optional but reduces rate-limit risk
#
# Features derived:
#   income                  — median household income (B19013)
#   commute_time            — mean commute minutes (B08136 / B08303)
#   pct_bachelors           — share of pop 25+ with bachelor's degree or higher (B15003)
#   racial_diversity_index  — 1 - Herfindahl index across race groups (B02001)

ACS_YEAR = 2022
ACS_VARS = [
    'B19013_001E',                                                          # median income
    'B08136_001E', 'B08303_001E',                                           # commute
    'B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E',            # education (BA+)
    'B15003_001E',                                                          # education total
    'B02001_001E', 'B02001_002E', 'B02001_003E', 'B02001_004E',            # race
    'B02001_005E', 'B02001_006E', 'B02001_007E', 'B02001_008E',
]
var_str   = ','.join(ACS_VARS)
key_param = f'&key={CENSUS_API_KEY}' if CENSUS_API_KEY else ''
url = (
    f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
    f'?get=NAME,{var_str}&for=zip%20code%20tabulation%20area:*{key_param}'
)

print('Fetching ACS data for all ZCTAs — this takes ~30–60 seconds...')
resp = requests.get(url, timeout=120)
resp.raise_for_status()
raw = resp.json()

headers  = raw[0]
zcta_col = headers.index('zip code tabulation area')

def safe_float(row, var):
    try:
        v = float(row[headers.index(var)])
        return None if v in (-666666666, -666666666.0) else v
    except Exception:
        return None

rows_acs = []
for row in raw[1:]:
    zcta = str(row[zcta_col]).zfill(5)

    income    = safe_float(row, 'B19013_001E')

    agg_min   = safe_float(row, 'B08136_001E')
    n_comm    = safe_float(row, 'B08303_001E')
    commute   = (agg_min / n_comm) if (agg_min and n_comm and n_comm > 0) else None

    total_edu = safe_float(row, 'B15003_001E')
    higher    = sum(safe_float(row, v) or 0
                    for v in ['B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E'])
    pct_ba    = (higher / total_edu) if (total_edu and total_edu > 0) else None

    total_race = safe_float(row, 'B02001_001E')
    if total_race and total_race > 0:
        race_vars = ['B02001_002E', 'B02001_003E', 'B02001_004E', 'B02001_005E',
                     'B02001_006E', 'B02001_007E', 'B02001_008E']
        sum_sq    = sum((safe_float(row, v) or 0) ** 2 for v in race_vars) / (total_race ** 2)
        diversity = 1.0 - sum_sq
    else:
        diversity = None

    rows_acs.append({
        'zip':                    zcta,
        'income':                 income,
        'commute_time':           commute,
        'pct_bachelors':          pct_ba,
        'racial_diversity_index': diversity,
    })

df_acs = pd.DataFrame(rows_acs)
df_acs.to_csv('/content/acs_features.csv', index=False)
print(f'\n✓ Saved /content/acs_features.csv — shape: {df_acs.shape}')
print(f'Non-null counts:\n{df_acs.notna().sum()}')

In [ ]:
# Cell 7 — Merge ZHVI targets + ACS features + FRED macros
#
# Join strategy:
#   ZHVI targets  ← ACS  : left join on zip (static, one row per ZIP)
#   ↑ result      ← FRED : merge_asof on date (backward — most recent macro for each row)

df_targets = pd.read_csv('/content/zhvi_targets.csv',  parse_dates=['date'])
df_fred    = pd.read_csv('/content/fred_macros.csv',    parse_dates=['date'])
df_acs     = pd.read_csv('/content/acs_features.csv',   dtype={'zip': str})

df_targets['zip'] = df_targets['zip'].astype(str).str.zfill(5)
df_acs['zip']     = df_acs['zip'].astype(str).str.zfill(5)

# ── Merge ACS (static per ZIP) ────────────────────────────────────────────────
df_merged = df_targets.merge(df_acs, on='zip', how='left')
print(f'After ACS merge   : {df_merged.shape}')

# ── Merge FRED macros by date (backward fill) ─────────────────────────────────
df_merged = df_merged.sort_values('date').reset_index(drop=True)
df_fred   = df_fred.sort_values('date').reset_index(drop=True)
df_merged = pd.merge_asof(df_merged, df_fred, on='date', direction='backward')
print(f'After FRED merge  : {df_merged.shape}')

df_merged.to_csv('/content/training_merged.csv', index=False)
print(f'\n✓ Saved /content/training_merged.csv')
print(f'\nNull counts:')
print(df_merged.isna().sum())
print(f'\nTarget distribution (appreciation %):')
print(df_merged['target'].describe().round(2))
df_merged.head()

In [ ]:
# Cell 8 — Preprocessing with 70/15/15 time-based split
#
# Target is now percentage points: e.g. 4.2 means +4.2% appreciation
# Filter bounds updated accordingly: drop extreme outliers outside [-50%, +100%]
# Time-based split preserves temporal order to prevent data leakage.

df = pd.read_csv('/content/training_merged.csv', parse_dates=['date'])

TARGET    = 'target'
DROP_COLS = ['date', 'baseline_zhvi']   # these are data-construction columns, not features
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
df = df.dropna(subset=[TARGET])

# Sanity bounds: [-50%, +100%] — consistent with the filter applied during target construction
before = len(df)
df = df[df[TARGET].between(-50.0, 100.0)]
print(f'Rows after outlier filter: {len(df):,}  (removed {before - len(df):,})')

# ── Drop all-null columns — imputer cannot handle them ────────────────────────
all_null = [c for c in df.columns if df[c].isna().all()]
if all_null:
    print(f'Dropping {len(all_null)} all-null columns: {all_null}')
    df = df.drop(columns=all_null)

CANDIDATE_CAT = ['zip', 'state', 'county']
CAT_COLS = [c for c in CANDIDATE_CAT if c in df.columns]
NUM_COLS = [c for c in df.columns if c not in set(CAT_COLS + [TARGET])]

print(f'\nTotal rows      : {len(df):,}')
print(f'Numeric features: {NUM_COLS}')
print(f'Categoric feats : {CAT_COLS}')

for c in CAT_COLS:
    df[c] = df[c].astype(str).fillna('unknown')

# ── Time-based 70 / 15 / 15 split (df already sorted by date from merge) ──────
n       = len(df)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

df_tr  = df.iloc[:tr_end].copy()
df_val = df.iloc[tr_end:val_end].copy()
df_te  = df.iloc[val_end:].copy()

print(f'\nSplit sizes:')
print(f'  Train      : {len(df_tr):>7,} rows  ({len(df_tr)/n*100:.1f}%)')
print(f'  Validation : {len(df_val):>7,} rows  ({len(df_val)/n*100:.1f}%)')
print(f'  Test       : {len(df_te):>7,} rows  ({len(df_te)/n*100:.1f}%)')

# ── Impute numerics (fit only on train) ───────────────────────────────────────
num_imputer = SimpleImputer(strategy='median')
X_tr_num  = pd.DataFrame(num_imputer.fit_transform(df_tr[NUM_COLS]),
                          columns=NUM_COLS, index=df_tr.index)
X_val_num = pd.DataFrame(num_imputer.transform(df_val[NUM_COLS]),
                          columns=NUM_COLS, index=df_val.index)
X_te_num  = pd.DataFrame(num_imputer.transform(df_te[NUM_COLS]),
                          columns=NUM_COLS, index=df_te.index)

# ── One-hot encode categoricals (fit only on train) ───────────────────────────
if CAT_COLS:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_cat  = pd.DataFrame(ohe.fit_transform(df_tr[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS), index=df_tr.index)
    X_val_cat = pd.DataFrame(ohe.transform(df_val[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS), index=df_val.index)
    X_te_cat  = pd.DataFrame(ohe.transform(df_te[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS), index=df_te.index)
    X_tr  = pd.concat([X_tr_num,  X_tr_cat],  axis=1)
    X_val = pd.concat([X_val_num, X_val_cat], axis=1)
    X_te  = pd.concat([X_te_num,  X_te_cat],  axis=1)
else:
    ohe   = None
    X_tr, X_val, X_te = X_tr_num, X_val_num, X_te_num

y_tr  = df_tr[TARGET].values
y_val = df_val[TARGET].values
y_te  = df_te[TARGET].values

FEATURE_NAMES = list(X_tr.columns)
print(f'\nFeature matrix shape: {X_tr.shape}')

# ── Save preprocessing metadata ───────────────────────────────────────────────
meta = {
    'num_cols':            NUM_COLS,
    'cat_cols':            CAT_COLS,
    'feature_names':       FEATURE_NAMES,
    'target_unit':         'percentage_points',   # e.g. 4.2 means +4.2%
    'num_imputer_medians': num_imputer.statistics_.tolist(),
    'ohe_categories':      {c: list(cats) for c, cats in zip(CAT_COLS, ohe.categories_)}
                           if ohe else {}
}
with open('/content/preprocess_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✓ Saved /content/preprocess_meta.json')

In [ ]:
# Cell 9 — Train XGBoost on GPU (A100)
#
# Targets are now real percentage points — expect sane MAE:
#   6-month horizon  : ~1–3 pp  (e.g. predicts 3.1% when actual is 4.2%)
#   12-month horizon : ~2–4 pp
#   36-month horizon : ~3–6 pp
# (vs. the broken pipeline's MAPE of 8–11 million %)

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:400])
print(f'XGBoost version: {xgboost.__version__}')

model = xgboost.XGBRegressor(
    tree_method='hist',           # XGBoost 2.0+ syntax (replaces 'gpu_hist')
    device='cuda',                # XGBoost 2.0+ syntax (replaces predictor='gpu_predictor')
    objective='reg:squarederror',
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    random_state=42
)

print('Training...')
model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=100
)
print(f'\nBest iteration: {model.best_iteration}')

# ── Evaluate ──────────────────────────────────────────────────────────────────
# Report MAE in percentage points.
# We intentionally skip MAPE here: when true appreciation is near 0%
# (e.g. flat markets), MAPE explodes arithmetically — it's not a useful
# metric for this target distribution. MAE in pp is interpretable and stable.
def evaluate(name, X, y):
    pred = model.predict(X)
    mae  = mean_absolute_error(y, pred)
    print(f'  {name:12s} — MAE: {mae:.2f} percentage points')

print('\n=== Evaluation ===')
evaluate('Train',      X_tr,  y_tr)
evaluate('Validation', X_val, y_val)
evaluate('Test',       X_te,  y_te)

# ── Save model ────────────────────────────────────────────────────────────────
model.save_model('/content/appreciation_xgb.json')
joblib.dump(model, '/content/appreciation_xgb.joblib')
print('\n✓ Saved /content/appreciation_xgb.json')
print('✓ Saved /content/appreciation_xgb.joblib')

In [ ]:
# Cell 10 — Scenario Perturbation
#
# Predict property appreciation under best / avg / worst macro scenarios.
# Perturbations shift mortgage_rate, fed_funds, and unemployment by fixed deltas.
#
# NOTE: model output is already in percentage points — do NOT multiply by 100
#       when displaying results. (The old pipeline output was a ratio and required *100.)

PERTURBATIONS = {
    'avg':   {},
    'best':  {'mortgage_rate': -1.0, 'fed_funds': -0.5, 'unemployment': -0.5},
    'worst': {'mortgage_rate': +1.0, 'fed_funds': +0.5, 'unemployment': +0.5},
}

def predict_scenarios(row: dict, horizon_months: int) -> dict:
    """
    Predict appreciation % under each macro scenario.

    Parameters
    ----------
    row : dict
        Feature dict for a single property/ZIP observation.
    horizon_months : int
        One of 6, 12, or 36.

    Returns
    -------
    dict  { 'best': float, 'avg': float, 'worst': float }
          Values are appreciation in percentage points.
    """
    results = {}
    for scenario_name, deltas in PERTURBATIONS.items():
        perturbed = row.copy()
        perturbed['horizon_months'] = horizon_months
        for macro_key, delta in deltas.items():
            if macro_key in perturbed and perturbed[macro_key] is not None:
                perturbed[macro_key] = float(perturbed[macro_key]) + delta

        row_df = pd.DataFrame([perturbed])

        # Align to training feature set
        for col in NUM_COLS:
            if col not in row_df.columns:
                row_df[col] = np.nan
        for col in CAT_COLS:
            if col not in row_df.columns:
                row_df[col] = 'unknown'

        X_num = pd.DataFrame(
            num_imputer.transform(row_df[NUM_COLS]), columns=NUM_COLS
        )
        if ohe and CAT_COLS:
            X_cat = pd.DataFrame(
                ohe.transform(row_df[CAT_COLS]),
                columns=ohe.get_feature_names_out(CAT_COLS)
            )
            X_scenario = pd.concat([X_num, X_cat], axis=1)
        else:
            X_scenario = X_num

        # Output is already percentage points — no *100 needed
        results[scenario_name] = float(model.predict(X_scenario)[0])

    return results


# ── Demonstration: run scenarios on a sample row from the test set ────────────
df_te_raw = pd.read_csv('/content/training_merged.csv', parse_dates=['date'])
df_te_raw = df_te_raw.drop(columns=[c for c in DROP_COLS if c in df_te_raw.columns])
sample    = df_te_raw.dropna(subset=['target']).iloc[-1].to_dict()

print('=== Sample row ===')
preview = ['zip', 'state', 'county', 'horizon_months',
           'income', 'pct_bachelors', 'mortgage_rate', 'fed_funds', 'unemployment']
for k in preview:
    if k in sample:
        print(f'  {k:28s}: {sample[k]}')

print('\n=== Scenario Predictions ===')
for horizon in [6, 12, 36]:
    preds = predict_scenarios(sample, horizon)
    print(f'\nHorizon: {horizon} months')
    for scenario, val in preds.items():
        print(f'  {scenario:6s}: {val:+.2f}% appreciation')

print('\n✓ Pipeline complete!')
print('Output files:')
for f in ['/content/appreciation_xgb.joblib',
          '/content/appreciation_xgb.json',
          '/content/preprocess_meta.json']:
    import os
    size = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f'  {f}  ({size:.0f} KB)')

In [ ]:
# Cell 11 — Download model artifacts
from google.colab import files
files.download('/content/appreciation_xgb.joblib')
files.download('/content/appreciation_xgb.json')
files.download('/content/preprocess_meta.json')